# RSNA Knee Abnormality Detection: a scalable baseline on frozen DINOv2 features

This notebook builds a baseline designed to be pushed further rather than a fixed, one-shot
pipeline. It uses the dataset properties established in the companion exploration notebook
(sparse direct labels, multilingual reports, variable series per study, missing laterality,
filename order not matching physical order) and turns them into a concrete, tunable pipeline.

## Why a frozen vision transformer instead of a CNN trained from scratch

The directly labelled subset of this dataset is small, on the order of a few dozen to a few
hundred studies out of several thousand. A convolutional network trained from scratch on that
much trustworthy signal, with the rest of the loss coming from a noisy keyword-derived weak
label, has very little to anchor its low-level visual features to and tends to fit the weak-
label heuristic rather than the underlying finding.

DINOv2 is a vision transformer trained with self-supervision on a very large, diverse image
corpus, without using any labels. Because that training signal comes entirely from the
structure of natural images rather than a specific label set, the resulting features transfer
unusually well to new domains, including grayscale medical imagery once it is replicated across
three channels, even though DINOv2 never saw an MRI during its own training. Using it as a
frozen feature extractor means the amount of *new* trainable capacity is just a small head, sized
to match how much trustworthy label signal actually exists here, while the heavy lifting of
turning pixels into a useful representation is done by weights that were never at risk of
overfitting this dataset's specific weak labels.

This also make the pipeline fast to iterate on: extracting features is done once and cached, so
changing the head architecture, the loss weighting, or the training schedule takes seconds
rather than a full pass over the imaging data. Unfreezing part or all of the backbone and
fine-tuning end to end is a natural next step once this pipeline is validated, and is set up as
a configuration flag rather than a rewrite; see the closing section.

## Pipeline

1. A single configuration block controlling image size, how many slices are sampled per plane,
   which backbone is used, whether it is frozen, and the training schedule.
2. Loading the competition files and selecting one representative series per anatomical plane
   for every study, as in the exploration notebook.
3. Reading several physically-ordered, correctly cropped slices per plane for every study, in
   parallel, and caching the resulting tensors to disk.
4. Extracting DINOv2 features for every slice, pooling across slices within a plane and
   concatenating across planes into one feature vector per study, cached to disk.
5. Building weak labels from the reports, overridden by direct annotation where available, with
   sample weighting that reflects how much each label can be trusted.
6. A report-hash-grouped validation split, to avoid leaking duplicate reports across folds.
7. Training a small multilayer perceptron head on top of the cached features.
8. Producing a submission from the test set, using the same feature pipeline.
9. A closing section that lays out, concretely, how to scale this baseline up: unfreezing the
   backbone, more slices, more folds and ensembling, laterality correction, and a stronger
   report labeller.

## A note on the offline scoring environment

This is a code competition: the notebook that gets submitted is re-run against the hidden test
set with internet access disabled. `pretrained=True` below will download DINOv2's weights from
the Hugging Face Hub, which works while editing the notebook interactively, since Kaggle
notebooks have internet access during editing, but will fail during a scored submission run.
Before submitting, the downloaded weights need to be attached as a Kaggle dataset and loaded
from disk instead; the loading function below is written to support both paths, and the
mechanics are explained where that function is defined.

## 1. Configuration

Every tunable choice in this notebook is collected here rather than scattered through the code
that uses it. This is what makes the notebook a starting point to scale up rather than a fixed
pipeline: increasing `SLICES_PER_PLANE`, swapping `BACKBONE_NAME` for a larger DINOv2 variant,
or flipping `FREEZE_BACKBONE` to fine-tune end to end are all one-line changes here, with the
rest of the notebook adapting automatically.

In [ ]:
CONFIG = {
    # --- imaging ---
    "IMG_SIZE": 224,            # square input size fed to the backbone
    "CROP_MM": 120.0,           # physical field of view cropped from the centre of each slice
    "SLICES_PER_PLANE": 3,      # slices sampled per selected series, spread through the stack
    "SLICE_QUANTILES": (0.35, 0.50, 0.65),  # where in the physically-ordered stack to sample

    # --- backbone ---
    "BACKBONE_NAME": "vit_small_patch14_dinov2",
    "PRETRAINED": True,         # set False only for a quick mechanics smoke test
    "LOCAL_BACKBONE_CHECKPOINT": None,  # path to a local .pth if internet is unavailable
    "FREEZE_BACKBONE": True,    # see section 9 for how and why to turn this off later

    # --- data pipeline ---
    "N_IO_WORKERS": 16,         # thread pool size for parallel DICOM reading
    "FEATURE_BATCH_SIZE": 64,   # batch size used only for the backbone forward pass

    # --- head training ---
    "HEAD_HIDDEN": 256,
    "HEAD_DROPOUT": 0.3,
    "HEAD_EPOCHS": 150,
    "HEAD_LR": 1e-3,
    "HEAD_WEIGHT_DECAY": 1e-4,
    "HEAD_BATCH_SIZE": 128,

    # --- validation ---
    "N_FOLDS": 5,
    "VALID_FOLD": 0,

    # --- caching, so repeated runs of this notebook do not redo the slow steps ---
    "CACHE_DIR": "/kaggle/working/cache",
}

import os
os.makedirs(CONFIG["CACHE_DIR"], exist_ok=True)
print("configuration set. slices per study per plane:", CONFIG["SLICES_PER_PLANE"])

## 2. Loading the data and choosing one series per plane

The competition root is located by walking `/kaggle/input`, since the mount path differs
depending on how the competition data was attached. Series selection follows the same rule
established in the exploration notebook: for each of the three anatomical planes, prefer a
fluid-sensitive series where one exists, since fluid-sensitive sequences carry information
relevant to more of the twelve findings than a purely structural sequence.

In [ ]:
import re
import time
import hashlib
import unicodedata
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
torch.manual_seed(0)
np.random.seed(0)

def find_competition_root(base="/kaggle/input"):
    for root, dirs, files in os.walk(base):
        if "train.csv" in files and ("train_series.csv" in files or "train_series" in dirs):
            return root
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
    raise FileNotFoundError(
        "Could not locate the competition files under /kaggle/input. "
        "Attach the RSNA Knee Abnormality Detection dataset to this notebook."
    )

ROOT = Path(find_competition_root())
train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")
train_series = pd.read_csv(ROOT / "train_series.csv")
test_series = pd.read_csv(ROOT / "test_series.csv")
sample_submission = pd.read_csv(ROOT / "sample_submission.csv")

TARGETS = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
has_full_labels = train[TARGETS].notna().all(axis=1)

print(f"studies total     : {len(train):,}  (directly labelled: {int(has_full_labels.sum())})")
print(f"targets           : {TARGETS}")
print(f"device            : {DEVICE}")

In [ ]:
def select_series_per_plane(series_df):
    # Returns, for every study, the chosen SeriesInstanceUID for each of the three planes.
    selected = {}
    fluid_col = "Fluid_Sensitive" if "Fluid_Sensitive" in series_df.columns else None
    for study_id, group in series_df.groupby("StudyInstanceUID"):
        choice = {}
        for plane in ("Sagittal", "Coronal", "Axial"):
            candidates = group[group["Anatomical_Plane"] == plane]
            if candidates.empty:
                continue
            ranked = candidates.sort_values(fluid_col, ascending=False) if fluid_col else candidates
            choice[plane] = ranked.iloc[0]["SeriesInstanceUID"]
        selected[study_id] = choice
    return selected

train_plane_series = select_series_per_plane(train_series)
test_plane_series = select_series_per_plane(test_series)

n_with_all_three = sum(1 for c in train_plane_series.values() if len(c) == 3)
print(f"training studies with all three planes available : {n_with_all_three}/{len(train_plane_series)}")

## 3. Reading several correctly-ordered, correctly-scaled slices per plane

Two corrections from the exploration notebook are applied here. Files within a series are
ordered by true physical position, computed from `ImagePositionPatient` and
`ImageOrientationPatient`, not by filename. Each slice is cropped to a fixed physical field of
view in millimetres using the DICOM pixel spacing, then resized to a fixed pixel size, so a
given anatomical structure occupies a comparable position and scale across studies regardless of
scanner resolution.

Rather than reading a single middle slice, `CONFIG["SLICES_PER_PLANE"]` slices are sampled at
fixed quantiles through each physically-ordered stack. This captures more of the study than a
single slice while staying inexpensive: the quantiles are fixed positions relative to the stack,
not a search over which slice looks most informative.

Reading is done with a thread pool. DICOM decoding is largely I/O bound and pydicom's decoding
step releases the interpreter lock for a meaningful share of its work, so threads give a real
speedup here on multi-core machines without the complexity of a process pool.

In [ ]:
def physically_ordered_files(series_dir):
    files = sorted(f for f in os.listdir(series_dir) if f.endswith(".dcm"))
    keyed = []
    for f in files:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, f), stop_before_pixels=True, force=True,
                                  specific_tags=["ImagePositionPatient", "ImageOrientationPatient"])
            orientation = np.asarray(ds.ImageOrientationPatient, dtype=float)
            position = np.asarray(ds.ImagePositionPatient, dtype=float)
            key = float(np.dot(position, np.cross(orientation[:3], orientation[3:])))
        except Exception:
            key = 0.0
        keyed.append((key, f))
    keyed.sort()
    return [f for _, f in keyed]

def read_slice(series_dir, filename, crop_mm, size):
    ds = pydicom.dcmread(os.path.join(series_dir, filename), force=True)
    array = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    array = array * slope + intercept

    try:
        spacing_mm_per_px = float(ds.PixelSpacing[0])
    except Exception:
        spacing_mm_per_px = crop_mm / max(array.shape)
    half_crop_px = max(1, int(round(crop_mm / spacing_mm_per_px / 2)))
    cy, cx = array.shape[0] // 2, array.shape[1] // 2
    y0, y1 = max(0, cy - half_crop_px), min(array.shape[0], cy + half_crop_px)
    x0, x1 = max(0, cx - half_crop_px), min(array.shape[1], cx + half_crop_px)
    crop = array[y0:y1, x0:x1]
    if crop.size == 0:
        crop = array

    low, high = np.percentile(crop, [1, 99])
    normalized = np.clip((crop - low) / max(high - low, 1e-6), 0, 1).astype(np.float32)
    y_idx = np.linspace(0, normalized.shape[0] - 1, size).astype(int)
    x_idx = np.linspace(0, normalized.shape[1] - 1, size).astype(int)
    return normalized[np.ix_(y_idx, x_idx)]

def study_slices(study_id, plane_series, series_root, cfg):
    # Returns an array of shape (3, K, size, size): plane x sampled-slice x H x W.
    # Missing planes, or a slice that fails to read, are returned as zero arrays rather than
    # raising, so a single bad file does not take down a whole batch.
    size = cfg["IMG_SIZE"]
    k = cfg["SLICES_PER_PLANE"]
    out = np.zeros((3, k, size, size), dtype=np.float32)
    choice = plane_series.get(study_id, {})
    for plane_idx, plane in enumerate(("Sagittal", "Coronal", "Axial")):
        series_id = choice.get(plane)
        if series_id is None:
            continue
        series_dir = os.path.join(series_root, str(study_id), str(series_id))
        try:
            files = physically_ordered_files(series_dir)
            if not files:
                continue
            for slice_idx, q in enumerate(cfg["SLICE_QUANTILES"][:k]):
                file_idx = min(len(files) - 1, max(0, int(round(q * (len(files) - 1)))))
                out[plane_idx, slice_idx] = read_slice(series_dir, files[file_idx], cfg["CROP_MM"], size)
        except Exception:
            continue
    return out

def build_slice_cache(study_ids, plane_series, series_root, cfg, cache_name):
    # Reads (or loads from cache) the (N, 3, K, size, size) slice tensor for a list of studies.
    cache_path = os.path.join(cfg["CACHE_DIR"], f"{cache_name}.npy")
    if os.path.exists(cache_path):
        print(f"loading cached slices from {cache_path}")
        return np.load(cache_path)

    size = cfg["IMG_SIZE"]
    k = cfg["SLICES_PER_PLANE"]
    result = np.zeros((len(study_ids), 3, k, size, size), dtype=np.float32)
    start = time.time()

    def _work(i):
        return i, study_slices(study_ids[i], plane_series, series_root, cfg)

    with ThreadPoolExecutor(max_workers=cfg["N_IO_WORKERS"]) as pool:
        for count, (i, arr) in enumerate(pool.map(_work, range(len(study_ids)))):
            result[i] = arr
            if (count + 1) % 500 == 0:
                print(f"  {count + 1}/{len(study_ids)} studies read ({time.time() - start:.0f}s elapsed)")

    np.save(cache_path, result)
    print(f"read {len(study_ids)} studies in {time.time() - start:.0f}s, cached to {cache_path}")
    return result

TRAIN_SERIES_ROOT = ROOT / "train_series"
TEST_SERIES_ROOT = ROOT / "test_series"

train_study_ids = train["StudyInstanceUID"].tolist()
test_study_ids = test["StudyInstanceUID"].tolist()

train_slices = build_slice_cache(train_study_ids, train_plane_series, TRAIN_SERIES_ROOT, CONFIG, "train_slices")
test_slices = build_slice_cache(test_study_ids, test_plane_series, TEST_SERIES_ROOT, CONFIG, "test_slices")
print("train_slices shape:", train_slices.shape, " test_slices shape:", test_slices.shape)

The cache written here is keyed only by a filename, not by the configuration that
produced it. Changing `IMG_SIZE`, `CROP_MM`, or `SLICES_PER_PLANE` after a first run requires
deleting the corresponding file under `CONFIG["CACHE_DIR"]`, or renaming the `cache_name`
arguments above, since the cache will otherwise silently return stale arrays at the old
resolution.

## 4. Loading DINOv2 and extracting features

The backbone is loaded through `timm`, which exposes DINOv2 alongside its ImageNet mean and
standard deviation for input normalisation. Two loading paths are supported: `pretrained=True`
downloads the weights from the Hugging Face Hub, which requires internet access and works while
editing the notebook; `LOCAL_BACKBONE_CHECKPOINT` loads a state dict from a local file instead,
for use once the weights have been downloaded once and attached to the notebook as a Kaggle
dataset, which is required before a scored submission run, since submissions execute with
internet disabled.

The backbone outputs a single pooled feature vector per input image rather than a spatial map,
by way of `num_classes=0`. `dynamic_img_size=True` allows an input size other than the 518x518
DINOv2 was originally trained at; 224 is used here to keep the feature extraction step fast,
which is a reasonable trade against resolution for a first baseline and is easy to raise later.

In [ ]:
import timm

def load_backbone(cfg):
    model_kwargs = dict(num_classes=0, img_size=cfg["IMG_SIZE"], dynamic_img_size=True)
    if cfg["LOCAL_BACKBONE_CHECKPOINT"]:
        model = timm.create_model(cfg["BACKBONE_NAME"], pretrained=False, **model_kwargs)
        state_dict = torch.load(cfg["LOCAL_BACKBONE_CHECKPOINT"], map_location="cpu")
        model.load_state_dict(state_dict, strict=False)
        print(f"loaded backbone weights from local checkpoint: {cfg['LOCAL_BACKBONE_CHECKPOINT']}")
    else:
        model = timm.create_model(cfg["BACKBONE_NAME"], pretrained=cfg["PRETRAINED"], **model_kwargs)
        if cfg["PRETRAINED"]:
            print("loaded pretrained backbone weights from the Hugging Face Hub "
                  "(requires internet; replace with LOCAL_BACKBONE_CHECKPOINT before submitting)")
        else:
            print("backbone initialised with random weights (PRETRAINED=False) — "
                  "for pipeline testing only, not for a real submission")
    data_cfg = timm.data.resolve_data_config({}, model=model)
    mean = torch.tensor(data_cfg["mean"]).view(1, 3, 1, 1)
    std = torch.tensor(data_cfg["std"]).view(1, 3, 1, 1)
    if cfg["FREEZE_BACKBONE"]:
        for p in model.parameters():
            p.requires_grad_(False)
        model.eval()
    return model.to(DEVICE), mean.to(DEVICE), std.to(DEVICE)

backbone, backbone_mean, backbone_std = load_backbone(CONFIG)
embed_dim = backbone.num_features
print(f"backbone embedding dimension : {embed_dim}, frozen: {CONFIG['FREEZE_BACKBONE']}")

Feature extraction processes one plane's slices at a time, across every study, so that a
single forward-pass batch is a stack of same-plane grayscale slices rather than a mix of
different planes. The single grayscale channel is replicated to three channels, since DINOv2
expects RGB input; this is a common and reasonable way to present grayscale medical images to a
network trained on natural images, though it does mean the three input channels are perfectly
correlated at the point they enter the network, unlike a genuine colour photograph.

Slices within a plane are pooled by averaging their embeddings, and the three plane embeddings
are concatenated rather than averaged together, since planes carry complementary rather than
redundant information, as the exploration notebook's example images showed directly.

In [ ]:
@torch.no_grad()
def extract_features(slice_tensor, backbone, mean, std, cfg, cache_path):
    # slice_tensor: (N, 3, K, size, size) float32 in [0, 1]. Returns (N, 3 * embed_dim).
    if os.path.exists(cache_path):
        print(f"loading cached features from {cache_path}")
        return np.load(cache_path)

    n_studies, n_planes, k, size, _ = slice_tensor.shape
    embed_dim = backbone.num_features
    batch_size = cfg["FEATURE_BATCH_SIZE"]
    use_amp = DEVICE.type == "cuda"

    # Flatten (plane, study, slice) into one long axis so the backbone always sees batches of
    # a fixed, configurable size, regardless of how many planes or slices-per-plane are used.
    flat_images = slice_tensor.transpose(1, 0, 2, 3, 4).reshape(n_planes * n_studies * k, size, size)
    flat_embeddings = np.zeros((flat_images.shape[0], embed_dim), dtype=np.float32)

    for start in range(0, flat_images.shape[0], batch_size):
        end = min(start + batch_size, flat_images.shape[0])
        batch = torch.from_numpy(flat_images[start:end]).unsqueeze(1).repeat(1, 3, 1, 1).to(DEVICE)
        batch = (batch - mean) / std
        with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
            out = backbone(batch)
        flat_embeddings[start:end] = out.float().cpu().numpy()

    flat_embeddings = flat_embeddings.reshape(n_planes, n_studies, k, embed_dim)
    plane_embeddings = flat_embeddings.mean(axis=2)          # average over the K slices
    study_features = plane_embeddings.transpose(1, 0, 2).reshape(n_studies, n_planes * embed_dim)

    np.save(cache_path, study_features)
    print(f"extracted features for {n_studies} studies, saved to {cache_path}")
    return study_features

train_feature_cache = os.path.join(CONFIG["CACHE_DIR"], "train_features.npy")
test_feature_cache = os.path.join(CONFIG["CACHE_DIR"], "test_features.npy")

X_train_feat = extract_features(train_slices, backbone, backbone_mean, backbone_std, CONFIG, train_feature_cache)
X_test_feat = extract_features(test_slices, backbone, backbone_mean, backbone_std, CONFIG, test_feature_cache)
print("X_train_feat shape:", X_train_feat.shape, " X_test_feat shape:", X_test_feat.shape)

Once this cell has been run once, both feature caches exist on disk, and every cell from
this point onward runs in seconds regardless of how large the imaging dataset is, since nothing
below this point touches a DICOM file or the backbone again.

## 5. Weak labels from the reports

Since the twelve target columns are empty for most studies, a first-pass label is extracted
from the free-text report using keyword patterns for each finding together with a nearby
negation check. This is intentionally a simple, conservative starting point rather than a
substitute for a proper multilingual medical text classifier: the exploration notebook found the
corpus spans multiple scripts and, within Latin script, more languages than are covered here, so
a meaningful share of reports will not match any pattern and are treated as unknown rather than
being defaulted to negative.

Every weak label is a soft value: 0.85 for a likely positive match, 0.15 for a likely negative
match, and 0.5, the least informative value under binary cross-entropy, for anything unmatched.
Directly labelled studies override these weak values entirely and receive a much higher sample
weight in training, since they are the only studies where the target is known rather than
inferred.

In [ ]:
FINDING_PATTERNS = {
    "ACL": r"\bacl\b|anterior cruciate|ligament.{0,15}crois|ligamento cruzado anterior|kreuzband",
    "MCL": r"\bmcl\b|medial collateral|colateral medial|innenband|ligamento colateral medial",
    "Medial Meniscus": r"medial meniscus|menisco medial|menisque interne|innenmeniskus",
    "Lateral Meniscus": r"lateral meniscus|menisco lateral|menisque externe|aussenmeniskus",
    "Medial OA": r"medial.{0,20}(arthrosis|osteoarthritis|compartment)|artrosis medial",
    "Lateral OA": r"lateral.{0,20}(arthrosis|osteoarthritis|compartment)|artrosis lateral",
    "PF OA": r"patellofemoral.{0,20}(arthrosis|osteoarthritis)|femoropatellar",
    "Effusion": r"effusion|joint fluid|derrame articular|epanchement|gelenkerguss",
    "Synovitis": r"synovitis|synovial|sinovitis|hoffa",
    "Baker's": r"baker|popliteal cyst|quiste popliteo|kyste poplite",
    "Contusion": r"contusion|bone bruise|bone marrow edema|edema.{0,10}medular|knochenmarks?odem",
    "Fracture": r"fracture|fractura|fraktur",
}
NEGATION_WORDS = r"(?:no|not|without|negative for|sin|sans|aucun|nicht|keine?|kein)\W{0,45}$"

def weak_labels_from_report(text):
    normalized = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii").lower()
    states = []
    for finding in TARGETS:
        pattern = FINDING_PATTERNS.get(finding)
        state = 0
        if pattern:
            for match in re.finditer(pattern, normalized):
                preceding_text = normalized[max(0, match.start() - 45):match.start()]
                state = -1 if re.search(NEGATION_WORDS, preceding_text) else 1
                if state == 1:
                    break
        states.append(state)
    return states

weak_states = np.array([weak_labels_from_report(t) for t in train["Report"].fillna("")], dtype=np.int8)
soft_targets = np.where(weak_states == 1, 0.85, np.where(weak_states == -1, 0.15, 0.5)).astype(np.float32)
sample_weights = np.where(weak_states == 0, 0.2, 1.0).astype(np.float32)

gold_mask = has_full_labels.to_numpy()
soft_targets[gold_mask] = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)
sample_weights[gold_mask] = 10.0

print(f"share of (study, finding) pairs the keyword search matched : {(weak_states != 0).mean() * 100:.1f}%")
print(f"directly labelled studies overriding the weak targets      : {int(gold_mask.sum())}")

## 6. A validation split that does not leak duplicate reports

Studies whose report text is byte-identical to another study's get the same weak targets, so a
random split can place near-duplicate examples on both sides of the train/validation boundary.
Grouping the split by a hash of the report text, rather than splitting per study independently,
keeps every member of a duplicate group on the same side. Studies with an empty report are
hashed individually so they are not all forced into one group.

In [ ]:
def report_group_id(text, study_id, n_folds):
    text = str(text).strip()
    key = text if text else f"__empty__{study_id}"
    return int(hashlib.md5(key.encode("utf-8")).hexdigest()[:8], 16) % n_folds

fold_id = np.array([report_group_id(t, s, CONFIG["N_FOLDS"])
                     for t, s in zip(train["Report"].fillna(""), train["StudyInstanceUID"])])
train_idx = np.flatnonzero(fold_id != CONFIG["VALID_FOLD"])
valid_idx = np.flatnonzero(fold_id == CONFIG["VALID_FOLD"])

print(f"training examples   : {len(train_idx):,}")
print(f"validation examples : {len(valid_idx):,}")
print(f"directly labelled studies in the validation fold : {int(gold_mask[valid_idx].sum())}")

## 7. Training the head

With features cached, the head is a small multilayer perceptron trained on top of them, which
makes iterating on the loss, the learning schedule, or the architecture fast, since no image
ever has to be re-read to try a change. The loss is a per-sample, per-finding weighted binary
cross-entropy against the soft targets: weak matches contribute a small amount, directly
labelled studies contribute a lot, and unmatched report or finding pairs contribute very little
since their target is the uninformative value of 0.5.

Macro-AUC is tracked on the validation fold in two forms: against the weak targets, which mostly
measures agreement with the report-parsing heuristic, and against whichever directly labelled
studies fall in the validation fold, which is the smaller but far more trustworthy number. Model
selection prefers the gold-subset AUC whenever the validation fold contains enough directly
labelled studies for it to be meaningful, and falls back to the weak-label AUC otherwise. If
neither ever produces a usable score, for example because a validation fold is small enough that
some finding has no positive example in it at all, the final epoch's weights are kept and the
notebook says so explicitly rather than failing.

In [ ]:
class KneeHead(nn.Module):
    def __init__(self, in_dim, n_targets, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_targets),
        )

    def forward(self, x):
        return self.net(x)

def to_tensor(array):
    return torch.tensor(array, dtype=torch.float32)

X_train_t = to_tensor(X_train_feat[train_idx]).to(DEVICE)
Y_train_t = to_tensor(soft_targets[train_idx]).to(DEVICE)
W_train_t = to_tensor(sample_weights[train_idx]).to(DEVICE)

X_valid_t = to_tensor(X_train_feat[valid_idx]).to(DEVICE)
Y_valid_weak = soft_targets[valid_idx]
valid_gold_mask = gold_mask[valid_idx]

gold_targets_full = np.full((len(train), len(TARGETS)), np.nan, dtype=np.float32)
gold_targets_full[gold_mask] = train.loc[has_full_labels, TARGETS].to_numpy(dtype=np.float32)
Y_valid_gold = gold_targets_full[valid_idx][valid_gold_mask]

def macro_auc(y_true, y_pred):
    scores = []
    for j in range(y_true.shape[1]):
        column = y_true[:, j]
        if len(np.unique(column)) < 2:
            continue
        scores.append(roc_auc_score(column, y_pred[:, j]))
    return float(np.mean(scores)) if scores else float("nan")

head = KneeHead(X_train_feat.shape[1], len(TARGETS), CONFIG["HEAD_HIDDEN"], CONFIG["HEAD_DROPOUT"]).to(DEVICE)
optimizer = torch.optim.AdamW(head.parameters(), lr=CONFIG["HEAD_LR"], weight_decay=CONFIG["HEAD_WEIGHT_DECAY"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["HEAD_EPOCHS"])

n_train = len(train_idx)
batch_size = min(CONFIG["HEAD_BATCH_SIZE"], n_train)
min_gold_for_selection = 5

best_score = -1.0
best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}  # always a valid fallback
used_fallback_state = True

for epoch in range(CONFIG["HEAD_EPOCHS"]):
    head.train()
    permutation = torch.randperm(n_train, device=DEVICE)
    epoch_loss = 0.0
    for start in range(0, n_train, batch_size):
        batch_idx = permutation[start:start + batch_size]
        logits = head(X_train_t[batch_idx])
        loss_per_element = F.binary_cross_entropy_with_logits(logits, Y_train_t[batch_idx], reduction="none")
        loss = (loss_per_element * W_train_t[batch_idx].unsqueeze(1)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(batch_idx)
    scheduler.step()
    epoch_loss /= n_train

    head.eval()
    with torch.no_grad():
        valid_pred = torch.sigmoid(head(X_valid_t)).cpu().numpy()
    weak_auc = macro_auc(Y_valid_weak, valid_pred)
    gold_auc = (macro_auc(Y_valid_gold, valid_pred[valid_gold_mask])
                if valid_gold_mask.sum() >= min_gold_for_selection else float("nan"))
    selection_score = gold_auc if not np.isnan(gold_auc) else weak_auc

    if not np.isnan(selection_score) and selection_score > best_score:
        best_score = selection_score
        best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        used_fallback_state = False

    if epoch % 10 == 0 or epoch == CONFIG["HEAD_EPOCHS"] - 1:
        gold_str = f"{gold_auc:.4f}" if not np.isnan(gold_auc) else "n/a"
        weak_str = f"{weak_auc:.4f}" if not np.isnan(weak_auc) else "n/a"
        print(f"epoch {epoch:3d}  loss {epoch_loss:.4f}  weak-val macro-AUC {weak_str}  "
              f"gold-val macro-AUC {gold_str}")

head.load_state_dict(best_state)
if used_fallback_state:
    print("\nno epoch produced a usable validation AUC (too few positive/negative examples "
          "in this validation fold for any finding); kept the head's initial weights. This is "
          "expected on a very small dataset and should not happen on the full competition data.")
else:
    print(f"\nbest selection score (gold AUC if available, else weak AUC) : {best_score:.4f}")

## 8. Producing predictions and a submission file

Test-set predictions reuse the same cached features built in section 4. Predictions are
converted to a percentile rank per finding before writing the submission file: since the
competition metric is ROC-AUC, which depends only on the ordering of predictions within each
finding rather than their absolute scale, rank conversion does not change the score but makes it
easy to sanity-check the output distribution across findings with very different prevalence.

In [ ]:
head.eval()
with torch.no_grad():
    test_pred = torch.sigmoid(head(to_tensor(X_test_feat).to(DEVICE))).cpu().numpy()

submission = sample_submission.copy()
rank_pct = pd.DataFrame(test_pred, columns=TARGETS).rank(pct=True)
submission[TARGETS] = rank_pct.to_numpy()
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("\nsubmission saved to /kaggle/working/submission.csv")

## 9. Scaling this baseline up

Every item below is a change within the structure already built, not a rewrite of it, and each
is a plausible next step in roughly the order of expected return for the effort involved.

**Unfreeze the backbone.** `FREEZE_BACKBONE` in the configuration controls this. Unfreezing
turns feature extraction from a one-time cached step into part of the training loop: the flat
feature cache no longer applies, `extract_features` would need to run inside the training loop
(or be replaced by a `Dataset`/`DataLoader` that reads slices on demand), and a much lower
learning rate, typically applied with layer-wise decay so earlier transformer blocks change less
than later ones, is needed to avoid destroying the pretrained representation. This is the single
change most likely to improve on this baseline, and also the most expensive one to run, which is
why it is not the default here.

**More slices, and more series per plane.** `SLICES_PER_PLANE` and `SLICE_QUANTILES` currently
sample a fixed few positions through one chosen series per plane. Sampling more positions, or
pooling across every slice in the stack rather than three fixed quantiles, gives the backbone
more of the study to draw on, at a roughly proportional cost in feature-extraction time. Using
more than one series per plane where a study has them, for example both a fat-suppressed and a
non-fat-suppressed sagittal series, would add complementary information the current single-slot-
per-plane design discards.

**Laterality correction.** None of the current pipeline knows which knee, left or right, is
being scanned, even though five of the twelve findings are defined relative to the body's
midline. Recovering laterality from the DICOM `Laterality` tag where present, and from the sign
of the physical slice position otherwise, and then mirroring left-knee images so that medial
structures always fall on the same side of the input, would let the model share structure across
left and right studies instead of implicitly having to learn both orientations unlabelled.

**A stronger report labeller.** The keyword and negation search in section 5 is a coarse
starting point that only covers a handful of languages well. A classifier trained on the small
set of directly labelled studies paired with their reports, or a general-purpose language model
prompted to extract structured findings, would likely produce cleaner weak targets and cover
more of the multilingual corpus than a fixed pattern list can.

**Cross-validate across all folds, and ensemble.** Only one of the `N_FOLDS` report-hash folds
is used here as a held-out validation set. Training a head for every fold and averaging their
test-set predictions, or at minimum checking that the gold-subset AUC is stable across folds
rather than reported from a single lucky or unlucky split, gives a more reliable estimate of how
the approach is likely to score and typically improves the final prediction as well.

**Test-time augmentation.** Averaging predictions from a slice and its horizontal mirror, or
from a couple of nearby crop centres, is inexpensive to add once the rest of the pipeline is
stable and generally gives a small, fairly reliable improvement.

**Attaching the backbone weights for submission.** Before submitting, run this notebook once
with an internet connection, save the backbone's state dict to a file, upload that file as a
Kaggle dataset, attach it to the submission notebook, and set `CONFIG["LOCAL_BACKBONE_CHECKPOINT"]`
to its path so the scored run never attempts a network download.